In [1]:
#for numerically named compounds, prefix is any text common to all BEFORE the number and suffix is common to all AFTER the number
#this is a template for our files that are all named "AcXXX_clust-X.log" or "AcXXX_conf-X.log"
prefix = ""
suffix = "_conf-"

#columns that provide atom mapping information are dropped
atom_columns_to_drop = ["Pt", "N1", "N2"]

#title of the column for the energy you want to use for boltzmann averaging and lowest E conformer determination
energy_col_header = "G(T)_spc(Hartree)"


energy_cutoff = 4.2 #specify energy cutoff in kcal/mol to remove conformers above this value before post-processing
verbose = False #set to true if you'd like to see info on the nunmber of conformers removed for each compound

### I will proceed with the clean-up of the dataframe

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("descriptors-conf-all.csv")

# --- 1. Configuration for your specific naming ---
separator = "-conf_"  # The part that identifies a conformer
energy_col_header = "G(T)_spc(Hartree)"
energy_cutoff = 4.2 
verbose = True
atom_columns_to_drop = ["Pt", "N1", "N2"]

# --- 2. Generate the Compound List ---
compound_list = []
for index, row in df.iterrows():
    log_file = row['log_name']
    # If the file has '-conf_', split and take the first part
    # If not (parent file), take the whole name
    compound = log_file.split(separator)[0]
    compound_list.append(compound)

compound_list = list(set(compound_list))
compound_list.sort()
print(f"Compounds to process: {compound_list}")

# --- 3. Initialize Master Dataframes ---
all_df_master = pd.DataFrame()
properties_df_master = pd.DataFrame()

for compound in compound_list:
    # Match the base name (e.g., 'imifoplatin' or 'imifoplatin-conf_1')
    valuesdf = df[df["log_name"].str.startswith(compound)].copy()
    
    # ADD THIS LINE: Force all column headers to be strings
    valuesdf.columns = valuesdf.columns.astype(str) 
    
    # Check if we accidentally grabbed similar names
    valuesdf = valuesdf[valuesdf["log_name"].apply(lambda x: x == compound or x.startswith(compound + separator))]

    # Column handling
    non_boltz_columns = ["G(Hartree)","∆G(Hartree)","∆G(kcal/mol)", "e^(-∆G/RT)","Mole Fraction"]
    reg_avg_columns = ['CPU_time_total(hours)', 'Wall_time_total(hours)']
    gv_extra_columns = ['E_spc (Hartree)', 'H_spc(Hartree)', 'T', 'T*S', 'T*qh_S', 'ZPE(Hartree)', 'qh_G(T)_spc(Hartree)', "G(T)_spc(Hartree)"]
    
    if str(energy_col_header) in gv_extra_columns:
        gv_extra_columns.remove(str(energy_col_header))

    # Boltzmann Statistics
    valuesdf["∆G(Hartree)"] = valuesdf[energy_col_header] - valuesdf[energy_col_header].min()
    valuesdf["∆G(kcal/mol)"] = valuesdf["∆G(Hartree)"] * 627.509
    valuesdf["e^(-∆G/RT)"] = np.exp((valuesdf["∆G(kcal/mol)"] * -1000) / (1.987204 * 298.15))
    valuesdf["Mole Fraction"] = valuesdf["e^(-∆G/RT)"] / valuesdf["e^(-∆G/RT)"].sum()
    
    initial = len(valuesdf.index)
    if verbose:
        print(f"\nProcessing: {compound}")
        print(f"Total conformers = {initial}")
        
    # Apply Cutoff
    valuesdf.drop(valuesdf[valuesdf["∆G(kcal/mol)"] >= energy_cutoff].index, inplace=True)
    valuesdf = valuesdf.reset_index(drop=True)
    final = len(valuesdf.index)
    
    if verbose:
        print(f"Conformers within {energy_cutoff} kcal/mol: {final}")

    # Initialize summary rows
    rows_dict = {
        "boltz": ["Boltzmann Averages"],
        "stdev": ["Boltzmann Standard Deviation"],
        "min": ["Ensemble Minimum"],
        "max": ["Ensemble Maximum"],
        "range": ["Ensemble Range"]
    }
    exclude_cols = []

    for column in valuesdf.columns:
        if "log_name" in column:
            exclude_cols.append(column)
        elif any(phrase in column for phrase in non_boltz_columns + gv_extra_columns):
            for k in rows_dict: rows_dict[k].append("")
        elif any(phrase in column for phrase in reg_avg_columns):
            rows_dict["boltz"].append(valuesdf[column].mean())
            for k in ["stdev", "min", "max", "range"]: rows_dict[k].append("")
        else:
            valuesdf[column] = pd.to_numeric(valuesdf[column], errors='coerce')
            b_avg = (valuesdf[column] * valuesdf["Mole Fraction"]).sum()
            rows_dict["boltz"].append(b_avg)
            rows_dict["min"].append(valuesdf[column].min())
            rows_dict["max"].append(valuesdf[column].max())
            rows_dict["range"].append(valuesdf[column].max() - valuesdf[column].min())

            # Weighted StDev
            delta_sq = (valuesdf[column] - b_avg)**2
            w = valuesdf["Mole Fraction"]
            if len(w) > 1:
                w_stdev = np.sqrt(np.sum(delta_sq * w) / (((len(w)-1)/len(w)) * np.sum(w)))
            else:
                w_stdev = 0
            rows_dict["stdev"].append(w_stdev)

    # Add calculated rows to dataframe
    for k in ["boltz", "stdev", "min", "max", "range"]:
        valuesdf.loc[len(valuesdf)] = rows_dict[k]

    # Reorder columns
    front_cols = ["log_name", energy_col_header,"∆G(Hartree)","∆G(kcal/mol)","e^(-∆G/RT)","Mole Fraction"]
    other_cols = [c for c in valuesdf.columns if c not in front_cols and c not in exclude_cols]
    valuesdf = valuesdf[front_cols + other_cols]

    # Lowest E Conformer Labeling
    low_e_idx = valuesdf[valuesdf["∆G(Hartree)"] == 0].index[0]
    new_row = valuesdf.loc[low_e_idx].copy()
    new_row['log_name'] = "Lowest E Conformer"
    valuesdf = pd.concat([valuesdf, pd.DataFrame([new_row])], ignore_index=True)

    # Specific Property Tracking (e.g., Vbur)
    vbur_col = "%Vbur_C4_3.0Å"
    if vbur_col in valuesdf.columns:
        try:
            min_val = valuesdf.loc[valuesdf["log_name"] == "Ensemble Minimum", vbur_col].values[0]
            v_idx = valuesdf[valuesdf[vbur_col] == min_val].index[0]
            v_row = valuesdf.loc[v_idx].copy()
            v_row['log_name'] = f"{vbur_col}_min_Conformer"
            valuesdf = pd.concat([valuesdf, pd.DataFrame([v_row])], ignore_index=True)
        except: pass

    # Build Master
    all_df_master = pd.concat([all_df_master, valuesdf])

    # Summary Properties Table (One row per drug)
    # We pull the values from the summary rows we just created at the bottom of valuesdf
    summary_vals = valuesdf.tail(7) # Includes Boltz, Stdev, Min, Max, Range, LowE, VburMin
    
    p_row = {'Compound_Name': compound}
    # Summary rows are indexed relative to the end
    # Boltz is -7, Stdev is -6, Min is -5, Max is -4, Range is -3, LowE is -2, VburMin is -1
    for col in other_cols:
        col_data = valuesdf[col].values
        p_row[f"{col}_Boltz"] = col_data[-7]
        p_row[f"{col}_Boltz_stdev"] = col_data[-6]
        p_row[f"{col}_min"] = col_data[-5]
        p_row[f"{col}_max"] = col_data[-4]
        p_row[f"{col}_range"] = col_data[-3]
        p_row[f"{col}_low_E"] = col_data[-2]

    properties_df_master = pd.concat([properties_df_master, pd.DataFrame([p_row])])

all_df_master.reset_index(drop=True, inplace=True)
properties_df_master.reset_index(drop=True, inplace=True)
display(properties_df_master)

Compounds to process: ['CHEMBL1743583', 'CHEMBL1743593', 'CHEMBL1743594', 'CHEMBL2068442', 'CHEMBL611190', 'carboplatin', 'cisplatin', 'imifoplatin', 'iproplatin', 'lobaplatin', 'nedaplatin', 'oxaliplatin', 'picoplatin', 'satraplatin']

Processing: CHEMBL1743583
Total conformers = 3
Conformers within 4.2 kcal/mol: 2

Processing: CHEMBL1743593
Total conformers = 11
Conformers within 4.2 kcal/mol: 7

Processing: CHEMBL1743594
Total conformers = 11
Conformers within 4.2 kcal/mol: 10

Processing: CHEMBL2068442
Total conformers = 4
Conformers within 4.2 kcal/mol: 3

Processing: CHEMBL611190
Total conformers = 11
Conformers within 4.2 kcal/mol: 11

Processing: carboplatin
Total conformers = 2
Conformers within 4.2 kcal/mol: 2

Processing: cisplatin
Total conformers = 3
Conformers within 4.2 kcal/mol: 3

Processing: imifoplatin
Total conformers = 4
Conformers within 4.2 kcal/mol: 1

Processing: iproplatin
Total conformers = 11
Conformers within 4.2 kcal/mol: 10

Processing: lobaplatin
Total c

,Compound_Name,HOMO_Boltz,HOMO_Boltz_stdev,HOMO_min,HOMO_max,HOMO_range,HOMO_low_E,LUMO_Boltz,LUMO_Boltz_stdev,LUMO_min,...,G_div_V_N_4(N)_min,G_div_V_N_4(N)_max,G_div_V_N_4(N)_range,G_div_V_N_4(N)_low_E,sum_frag_G_div_V_N_Boltz,sum_frag_G_div_V_N_Boltz_stdev,sum_frag_G_div_V_N_min,sum_frag_G_div_V_N_max,sum_frag_G_div_V_N_range,sum_frag_G_div_V_N_low_E
0,CHEMBL1743583,-0.52242,-0.541884,2.119619e-03,-0.54203,-0.52242,0.01961,-0.36502,-0.356649,0.000908,...,1.157993e-08,0.000051,0.000051,1.000000e-07,0.000051,0.000051,1.157993e-08,0.000051,0.000051,1.000000e-07
1,CHEMBL1743593,-0.53536,-0.535313,7.785028e-05,-0.53633,-0.53530,0.00103,-0.32763,-0.326923,0.000045,...,1.847936e-09,0.000051,0.000051,0.000000e+00,0.000051,0.000051,1.847936e-09,0.000051,0.000051,0.000000e+00
2,CHEMBL1743594,-0.49572,-0.490840,2.802763e-03,-0.49572,-0.47980,0.01592,-0.33121,-0.328680,0.000939,...,5.952115e-08,0.000051,0.000051,2.000000e-07,0.000051,0.000051,5.952115e-08,0.000051,0.000051,2.000000e-07
3,CHEMBL2068442,-0.21650,-0.216606,9.138974e-05,-0.21667,-0.21650,0.00017,-0.04214,-0.042038,0.000073,...,4.343506e-09,0.000051,0.000051,0.000000e+00,0.000051,0.000051,4.343506e-09,0.000051,0.000051,0.000000e+00
4,CHEMBL611190,-0.36336,-0.363982,8.831147e-04,-0.36628,-0.36289,0.00339,-0.17967,-0.181061,0.001474,...,2.542916e-08,0.000051,0.000051,1.000000e-07,0.000051,0.000051,2.542916e-08,0.000051,0.000051,1.000000e-07
5,carboplatin,-0.22373,-0.223735,7.071043e-06,-0.22374,-0.22373,0.00001,-0.05916,-0.059155,0.000007,...,9.583084e-21,0.000051,0.000051,0.000000e+00,0.000051,0.000051,9.583084e-21,0.000051,0.000051,0.000000e+00
6,cisplatin,-0.22515,-0.225167,3.801209e-05,-0.22521,-0.22514,0.00007,-0.05984,-0.059867,0.000074,...,0.000000e+00,0.000051,0.000051,0.000000e+00,0.000051,0.000051,0.000000e+00,0.000051,0.000051,0.000000e+00
7,imifoplatin,-0.23549,-0.234928,0.000000e+00,-0.23549,-0.23549,0.00000,-0.04958,-0.049462,0.000000,...,0.000000e+00,0.000051,0.000051,0.000000e+00,0.000051,0.000051,0.000000e+00,0.000051,0.000051,0.000000e+00
8,iproplatin,-0.23430,-0.233864,1.747055e-03,-0.23691,-0.23107,0.00584,-0.09936,-0.095585,0.001058,...,5.650188e-08,0.000051,0.000051,2.000000e-07,0.000051,0.000051,5.650188e-08,0.000051,0.000051,2.000000e-07
9,lobaplatin,-0.18848,-0.189917,1.079752e-03,-0.19066,-0.18848,0.00218,-0.03463,-0.028601,0.004543,...,0.000000e+00,0.000051,0.000051,0.000000e+00,0.000051,0.000051,0.000000e+00,0.000051,0.000051,0.000000e+00


In [3]:
# Assuming your dataframe is named properties_df_master
start_idx = properties_df_master.columns.get_loc('Compound_Name')
end_idx = properties_df_master.columns.get_loc('HOMO_Boltz')

# Drop the columns strictly between the two boundaries
properties_df_master.drop(properties_df_master.columns[start_idx + 1 : end_idx], axis=1, inplace=True)

# Verify it worked
display(properties_df_master)

,Compound_Name,HOMO_Boltz,HOMO_Boltz_stdev,HOMO_min,HOMO_max,HOMO_range,HOMO_low_E,LUMO_Boltz,LUMO_Boltz_stdev,LUMO_min,...,G_div_V_N_4(N)_min,G_div_V_N_4(N)_max,G_div_V_N_4(N)_range,G_div_V_N_4(N)_low_E,sum_frag_G_div_V_N_Boltz,sum_frag_G_div_V_N_Boltz_stdev,sum_frag_G_div_V_N_min,sum_frag_G_div_V_N_max,sum_frag_G_div_V_N_range,sum_frag_G_div_V_N_low_E
0,CHEMBL1743583,-0.52242,-0.541884,2.119619e-03,-0.54203,-0.52242,0.01961,-0.36502,-0.356649,0.000908,...,1.157993e-08,0.000051,0.000051,1.000000e-07,0.000051,0.000051,1.157993e-08,0.000051,0.000051,1.000000e-07
1,CHEMBL1743593,-0.53536,-0.535313,7.785028e-05,-0.53633,-0.53530,0.00103,-0.32763,-0.326923,0.000045,...,1.847936e-09,0.000051,0.000051,0.000000e+00,0.000051,0.000051,1.847936e-09,0.000051,0.000051,0.000000e+00
2,CHEMBL1743594,-0.49572,-0.490840,2.802763e-03,-0.49572,-0.47980,0.01592,-0.33121,-0.328680,0.000939,...,5.952115e-08,0.000051,0.000051,2.000000e-07,0.000051,0.000051,5.952115e-08,0.000051,0.000051,2.000000e-07
3,CHEMBL2068442,-0.21650,-0.216606,9.138974e-05,-0.21667,-0.21650,0.00017,-0.04214,-0.042038,0.000073,...,4.343506e-09,0.000051,0.000051,0.000000e+00,0.000051,0.000051,4.343506e-09,0.000051,0.000051,0.000000e+00
4,CHEMBL611190,-0.36336,-0.363982,8.831147e-04,-0.36628,-0.36289,0.00339,-0.17967,-0.181061,0.001474,...,2.542916e-08,0.000051,0.000051,1.000000e-07,0.000051,0.000051,2.542916e-08,0.000051,0.000051,1.000000e-07
5,carboplatin,-0.22373,-0.223735,7.071043e-06,-0.22374,-0.22373,0.00001,-0.05916,-0.059155,0.000007,...,9.583084e-21,0.000051,0.000051,0.000000e+00,0.000051,0.000051,9.583084e-21,0.000051,0.000051,0.000000e+00
6,cisplatin,-0.22515,-0.225167,3.801209e-05,-0.22521,-0.22514,0.00007,-0.05984,-0.059867,0.000074,...,0.000000e+00,0.000051,0.000051,0.000000e+00,0.000051,0.000051,0.000000e+00,0.000051,0.000051,0.000000e+00
7,imifoplatin,-0.23549,-0.234928,0.000000e+00,-0.23549,-0.23549,0.00000,-0.04958,-0.049462,0.000000,...,0.000000e+00,0.000051,0.000051,0.000000e+00,0.000051,0.000051,0.000000e+00,0.000051,0.000051,0.000000e+00
8,iproplatin,-0.23430,-0.233864,1.747055e-03,-0.23691,-0.23107,0.00584,-0.09936,-0.095585,0.001058,...,5.650188e-08,0.000051,0.000051,2.000000e-07,0.000051,0.000051,5.650188e-08,0.000051,0.000051,2.000000e-07
9,lobaplatin,-0.18848,-0.189917,1.079752e-03,-0.19066,-0.18848,0.00218,-0.03463,-0.028601,0.004543,...,0.000000e+00,0.000051,0.000051,0.000000e+00,0.000051,0.000051,0.000000e+00,0.000051,0.000051,0.000000e+00


In [4]:
properties_df_master.to_csv("descriptors-all.csv", index=False)